# Phase 1 — Data Collection & Audit

This notebook:
1. Downloads ~400 robot images using `icrawler` (run locally on Mac)
2. Auto-annotates all images using Autodistill — Florence-2 + SAM 2 (run on Colab T4)
3. Uploads images + auto-generated labels to Roboflow for review
4. After review, exports the final dataset back in YOLO format
5. Runs mandatory VIZ 1.A / 1.B / 1.C audit checks

**Target:** ≥ 300 annotated images, ≥ 25 instances per class in train set.

---
### Workflow
```
Cells A1–A3  [Mac, CPU]     → download + deduplicate ~400 images locally
Cell  B1–B2  [Colab, T4]    → auto-annotate all images with Autodistill
Cell  B3     [Mac, CPU]     → upload images + labels to Roboflow for review
                  ↓
              Go to Roboflow → review boxes, fix mistakes, approve images
                  ↓
Cell  C1–C2  [Mac, CPU]     → export final dataset from Roboflow → local
Cells D1–D3  [Mac, CPU]     → generate audit visualizations
```

In [1]:
import os, sys, random, collections, shutil, hashlib
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_DIR = '/Users/vineetjangir/robust&uncertainty-aware-robot-perception'
RAW_DIR     = f'{PROJECT_DIR}/data/raw'
DATA_DIR    = f'{PROJECT_DIR}/data/annotated'
RESULTS_DIR = f'{PROJECT_DIR}/results/figures'

for d in [RAW_DIR, RESULTS_DIR,
          f'{DATA_DIR}/images/train', f'{DATA_DIR}/images/val',
          f'{DATA_DIR}/images/test',  f'{DATA_DIR}/images/calibration',
          f'{DATA_DIR}/labels/train', f'{DATA_DIR}/labels/val',
          f'{DATA_DIR}/labels/test',  f'{DATA_DIR}/labels/calibration']:
    os.makedirs(d, exist_ok=True)

CLASS_NAMES = ['arm', 'leg', 'torso', 'head', 'sensor']
print('Directories ready.')

Directories ready.


## Part A — Download raw robot images

We use `icrawler` to pull images from Bing across ~20 search queries covering diverse humanoid robots.
Each query fetches ~25 images → ~500 total before deduplication → keep best 350+.

**Runtime: ~5–10 minutes**

In [2]:
# Cell A1 — Install icrawler
!pip install -q icrawler

In [ ]:
# Cell A2 — Download images across diverse robot search queries
from icrawler.builtin import BingImageCrawler

# ~25 images per query × 20 queries = ~500 raw images
SEARCH_QUERIES = [
    # Full-body bipedal humanoids (best for all 5 classes)
    'Boston Dynamics Atlas humanoid robot full body',
    'ASIMO Honda robot walking full body',
    'Unitree H1 humanoid robot',
    'Agility Robotics Digit robot',
    'Figure AI humanoid robot',
    'Fourier Intelligence humanoid robot',
    'UBTECH Walker humanoid robot',
    'Sanctuary AI Phoenix humanoid robot',
    # Specific body parts visible
    'humanoid robot arm gripper close up',
    'humanoid robot legs walking',
    'robot torso chest body',
    'robot head sensor camera',
    # Industrial/research robots with clear limbs
    'NAO robot humanoid full body',
    'Pepper robot Softbank full body',
    'Spot robot Boston Dynamics legs',
    'Valkyrie NASA humanoid robot',
    'iCub humanoid robot',
    'ROMEO robot humanoid',
    # More variety
    'humanoid robot laboratory research',
    'bipedal robot competition DARPA',
]

IMAGES_PER_QUERY = 25

for i, query in enumerate(SEARCH_QUERIES):
    # Use a safe folder name derived from query index
    out_dir = os.path.join(RAW_DIR, f'q{i:02d}')
    os.makedirs(out_dir, exist_ok=True)

    # Skip if already downloaded (re-run safe)
    existing = [f for f in os.listdir(out_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
    if len(existing) >= IMAGES_PER_QUERY * 0.8:
        print(f'[{i:02d}] Already have {len(existing)} images, skipping: {query[:50]}')
        continue

    print(f'[{i:02d}] Downloading: {query[:60]}')
    try:
        crawler = BingImageCrawler(storage={'root_dir': out_dir})
        crawler.crawl(keyword=query, max_num=IMAGES_PER_QUERY, min_size=(200, 200))
        downloaded = len([f for f in os.listdir(out_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
        print(f'      → {downloaded} images saved')
    except Exception as e:
        print(f'      ERROR: {e}')

total = sum(
    len([f for f in os.listdir(os.path.join(RAW_DIR, d)) if f.endswith(('.jpg', '.png', '.jpeg'))])
    for d in os.listdir(RAW_DIR) if os.path.isdir(os.path.join(RAW_DIR, d))
)
print(f'\nTotal raw images downloaded: {total}')

In [5]:
# Cell A3 — Deduplicate by MD5 hash + filter corrupt/tiny images
# Flattens all query subdirs into RAW_DIR/all_unique/

ALL_UNIQUE_DIR = os.path.join(RAW_DIR, 'all_unique')
os.makedirs(ALL_UNIQUE_DIR, exist_ok=True)

seen_hashes = set()
kept, skipped_dup, skipped_bad = 0, 0, 0

for subdir in sorted(os.listdir(RAW_DIR)):
    subpath = os.path.join(RAW_DIR, subdir)
    if not os.path.isdir(subpath) or subdir == 'all_unique':
        continue
    for fname in os.listdir(subpath):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        fpath = os.path.join(subpath, fname)
        # Check image loads and is large enough
        img = cv2.imread(fpath)
        if img is None or img.shape[0] < 100 or img.shape[1] < 100:
            skipped_bad += 1
            continue
        # Deduplicate by MD5
        with open(fpath, 'rb') as f:
            h = hashlib.md5(f.read()).hexdigest()
        if h in seen_hashes:
            skipped_dup += 1
            continue
        seen_hashes.add(h)
        # Copy with unique name: q00_000001.jpg etc
        ext = os.path.splitext(fname)[1].lower()
        dest_name = f'{subdir}_{fname}'
        shutil.copy2(fpath, os.path.join(ALL_UNIQUE_DIR, dest_name))
        kept += 1

print(f'Kept:            {kept}')
print(f'Skipped (dup):   {skipped_dup}')
print(f'Skipped (bad):   {skipped_bad}')
print(f'\nUnique usable images: {kept}')
if kept < 300:
    print('⚠️  Below 300 — consider adding more search queries in Cell A2 and re-running.')
else:
    print('✓  Enough images to proceed to annotation.')

Kept:            416
Skipped (dup):   3
Skipped (bad):   0

Unique usable images: 416
✓  Enough images to proceed to annotation.


## Part B — Auto-annotate all images with Autodistill (Colab T4)

**Switch to Google Colab with a T4 GPU runtime for cells B1–B2.**

### Before running:
1. Upload your `data/raw/all_unique/` folder to Google Drive at `My Drive/robot-perception/data/raw/all_unique/`
2. Open this notebook in Colab → Runtime → Change runtime type → T4 GPU
3. Run B1 then B2 (~3–5 min total)
4. Download the output `data/auto_labeled/` folder back to your Mac

**Then switch back to your Mac for B3 onwards.**

In [ ]:
# Cell B1 — Install Autodistill + Grounding DINO (run on Colab T4)
# Grounding DINO does NOT require flash-attn, so this installs cleanly on Colab.
!pip install -q autodistill autodistill-grounding-dino supervision roboflow

In [ ]:
# Cell B2 — Auto-annotate all images with Grounding DINO (Colab T4)
# Runtime: ~3-6 minutes for 400 images on T4 GPU

from google.colab import drive
drive.mount('/content/drive')

from autodistill_grounding_dino import GroundingDINO
from autodistill.detection import CaptionOntology
import os

COLAB_INPUT  = '/content/drive/MyDrive/robot-perception/data/raw/all_unique'
COLAB_OUTPUT = '/content/drive/MyDrive/robot-perception/data/auto_labeled'

# Ontology: descriptive text prompt -> your class name
base_model = GroundingDINO(
    ontology=CaptionOntology({
        'robotic arm of a humanoid robot': 'arm',
        'robot forearm or gripper':        'arm',
        'robotic leg of a humanoid robot': 'leg',
        'robot foot or lower leg':         'leg',
        'humanoid robot torso or body':    'torso',
        'robot chest or trunk':            'torso',
        'robot head unit':                 'head',
        'camera sensor on robot head':     'sensor',
        'depth sensor or lidar on robot':  'sensor',
    })
)

print(f'Auto-annotating images in: {COLAB_INPUT}')
print('Running on T4 GPU...')

base_model.label(
    input_folder=COLAB_INPUT,
    extension='.jpg',
    output_folder=COLAB_OUTPUT,
)

label_files = [f for f in os.listdir(f'{COLAB_OUTPUT}/labels') if f.endswith('.txt')]
annotated = sum(1 for f in label_files if os.path.getsize(f'{COLAB_OUTPUT}/labels/{f}') > 0)
print(f'\nDone. {annotated}/{len(label_files)} images have detections.')
print(f'Images with no detections: {len(label_files)-annotated}  <- needs manual boxes in Roboflow')
print(f'\nNow download {COLAB_OUTPUT} to your Mac and continue with cell B3.')

## ⏸️  PAUSE — Download auto_labeled folder to your Mac

Download `My Drive/robot-perception/data/auto_labeled/` from Google Drive to:
`/Users/vineetjangir/robust&uncertainty-aware-robot-perception/data/auto_labeled/`

Then run cell B3 below (back on your Mac).

In [ ]:
# Cell B3 — Upload images + auto-generated labels to Roboflow for review (run on Mac)
# Switch back to Mac kernel before running this

import os, glob
from roboflow import Roboflow

ROBOFLOW_API_KEY = 'YOUR_API_KEY_HERE'  # <- paste your Roboflow API key
MY_WORKSPACE     = 'vineet-jangir'
MY_PROJECT       = 'robot-parts-vbghi'

AUTO_LABELED_DIR = f'{PROJECT_DIR}/data/auto_labeled'

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(MY_WORKSPACE).project(MY_PROJECT)

image_files = glob.glob(f'{AUTO_LABELED_DIR}/images/*.jpg') + \
              glob.glob(f'{AUTO_LABELED_DIR}/images/*.jpeg') + \
              glob.glob(f'{AUTO_LABELED_DIR}/images/*.png')

print(f'Uploading {len(image_files)} images + labels to Roboflow...')
failed = []
for i, img_path in enumerate(image_files):
    stem = os.path.splitext(os.path.basename(img_path))[0]
    label_path = f'{AUTO_LABELED_DIR}/labels/{stem}.txt'
    try:
        if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
            project.upload(image_path=img_path, annotation_path=label_path,
                           annotation_labelmap={0:'arm',1:'leg',2:'torso',3:'head',4:'sensor'})
        else:
            project.upload(image_path=img_path)  # upload image only, annotate manually
        if (i+1) % 50 == 0:
            print(f'  {i+1}/{len(image_files)}...')
    except Exception as e:
        failed.append(img_path)

print(f'\nUploaded: {len(image_files)-len(failed)}, Failed: {len(failed)}')
print(f'\n-> Now go to app.roboflow.com/{MY_WORKSPACE}/{MY_PROJECT}')
print('   Review boxes: fix wrong labels, add missing boxes, delete bad images.')

## ⏸️  PAUSE — Review annotations in Roboflow

Go to your Roboflow project → **Annotate** → open each image and:
- Fix wrong class labels
- Tighten loose boxes
- Add missing boxes (especially `sensor`)
- Delete images that are clearly wrong (industrial arms, wheeled robots, etc.)

**Keyboard shortcuts:**
| Key | Action |
|---|---|
| `A` | Approve image |
| `R` | Reject (fix later) |
| `→` | Next image |

When done reviewing all images:
1. Click **Generate Version**
2. Split: 70% train / 15% val / 15% test
3. No augmentations
4. Click **Generate**
5. Come back and run Part C

## Part C — Export final dataset from Roboflow → local

In [ ]:
# Cell C1 — Download annotated dataset from Roboflow
from roboflow import Roboflow

ROBOFLOW_API_KEY = 'YOUR_API_KEY_HERE'  # <- paste your key
MY_WORKSPACE     = 'vineet-jangir'
MY_PROJECT       = 'robot-parts-vbghi'
MY_VERSION       = 1  # increment if you regenerated

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
dataset = rf.workspace(MY_WORKSPACE).project(MY_PROJECT).version(MY_VERSION).download(
    'yolov8',
    location=f'{PROJECT_DIR}/data/roboflow_export'
)
print('Dataset downloaded.')

In [ ]:
# Cell C2 — Reorganize into project folder structure + carve calibration set
import os, shutil, random

EXPORT_DIR = f'{PROJECT_DIR}/data/roboflow_export'
SPLIT_MAP  = {'train': 'train', 'valid': 'val', 'test': 'test'}

for rf_split, our_split in SPLIT_MAP.items():
    for kind in ('images', 'labels'):
        src = os.path.join(EXPORT_DIR, rf_split, kind)
        dst = os.path.join(DATA_DIR, kind, our_split)
        if not os.path.exists(src):
            print(f'  Missing: {src}')
            continue
        os.makedirs(dst, exist_ok=True)
        files = os.listdir(src)
        for f in files:
            shutil.copy2(os.path.join(src, f), os.path.join(dst, f))
        print(f'  {rf_split}/{kind} -> {kind}/{our_split}  ({len(files)} files)')

# Carve calibration set (100 images from train) — run once only
CALIB_N    = 100
train_imgs = [f for f in os.listdir(f'{DATA_DIR}/images/train')
              if f.lower().endswith(('.jpg','.jpeg','.png'))]
existing   = [f for f in os.listdir(f'{DATA_DIR}/images/calibration')
              if f.lower().endswith(('.jpg','.jpeg','.png'))]

if existing:
    print(f'\nCalibration set already exists ({len(existing)} images). Skipping.')
elif len(train_imgs) < CALIB_N + 50:
    print(f'\nNot enough train images ({len(train_imgs)}) to carve {CALIB_N}.')
else:
    random.seed(42)
    calib = random.sample(train_imgs, CALIB_N)
    for fname in calib:
        stem = os.path.splitext(fname)[0]
        shutil.move(f'{DATA_DIR}/images/train/{fname}',
                    f'{DATA_DIR}/images/calibration/{fname}')
        lbl = f'{DATA_DIR}/labels/train/{stem}.txt'
        if os.path.exists(lbl):
            shutil.move(lbl, f'{DATA_DIR}/labels/calibration/{stem}.txt')
    print(f'\nCalibration: {CALIB_N} images')
    print(f'Train remaining: {len(train_imgs)-CALIB_N} images')

for split in ('train','val','test','calibration'):
    n = len([f for f in os.listdir(f'{DATA_DIR}/images/{split}')
             if f.lower().endswith(('.jpg','.jpeg','.png'))])
    print(f'  {split:12s}: {n}')
total = sum(len([f for f in os.listdir(f'{DATA_DIR}/images/{s}')
                 if f.lower().endswith(('.jpg','.jpeg','.png'))])
            for s in ('train','val','test','calibration'))
print(f'  {"TOTAL":12s}: {total}')
print('OK' if total >= 300 else 'WARNING: below 300 images')

## VIZ 1.A — Class distribution bar chart

In [ ]:
def count_class_distribution(labels_dir):
    counts = collections.Counter()
    if not os.path.exists(labels_dir):
        return counts
    for label_file in os.listdir(labels_dir):
        if not label_file.endswith('.txt'): continue
        with open(os.path.join(labels_dir, label_file)) as f:
            for line in f:
                line = line.strip()
                if line:
                    counts[int(line.split()[0])] += 1
    return counts

train_counts = count_class_distribution(f'{DATA_DIR}/labels/train')
bar_values = [train_counts[i] for i in range(5)]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(CLASS_NAMES, bar_values, color='steelblue', edgecolor='white')
ax.axhline(y=25, color='red', linestyle='--', label='min threshold (25)')
for bar, count in zip(bars, bar_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(count), ha='center', va='bottom', fontsize=11)
ax.set_ylabel('Annotated instances (train)')
ax.set_title('VIZ 1.A — Class distribution (training set)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/viz1a_class_distribution.png', dpi=150)
plt.show()
print('⚠️  Any bar below red line = not enough data. Fix before training.')

## VIZ 1.B — 20 random annotated images

In [ ]:
def draw_annotations(img_path, label_path):
    img = cv2.imread(img_path)
    if img is None: return np.zeros((224,224,3), dtype=np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    COLORS_BGR = [(255,80,80),(80,200,80),(80,80,255),(255,200,0),(200,80,255)]
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                cls = int(parts[0])
                cx,cy,bw,bh = float(parts[1]),float(parts[2]),float(parts[3]),float(parts[4])
                x1=int((cx-bw/2)*w); y1=int((cy-bh/2)*h)
                x2=int((cx+bw/2)*w); y2=int((cy+bh/2)*h)
                color = COLORS_BGR[cls % 5]
                cv2.rectangle(img,(x1,y1),(x2,y2),color,2)
                cv2.putText(img,CLASS_NAMES[cls],(x1,max(y1-6,0)),
                            cv2.FONT_HERSHEY_SIMPLEX,0.55,color,2)
    return img

img_dir = f'{DATA_DIR}/images/train'
lbl_dir = f'{DATA_DIR}/labels/train'
img_files = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
sample = random.sample(img_files, min(20, len(img_files)))

fig, axes = plt.subplots(4, 5, figsize=(20, 16))
for ax, fname in zip(axes.flatten(), sample):
    stem = os.path.splitext(fname)[0]
    img = draw_annotations(f'{img_dir}/{fname}', f'{lbl_dir}/{stem}.txt')
    ax.imshow(img); ax.set_title(fname[:20], fontsize=8); ax.axis('off')
for ax in axes.flatten()[len(sample):]: ax.axis('off')
plt.suptitle('VIZ 1.B — 20 random annotated training images', fontsize=14)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/viz1b_annotation_sample.png', dpi=120)
plt.show()
print('⚠️  MANUAL CHECK: tight boxes? correct class labels? No whole-image boxes?')

## VIZ 1.C — Bounding box size distribution

In [ ]:
widths, heights = [], []
for lf in os.listdir(f'{DATA_DIR}/labels/train'):
    if not lf.endswith('.txt'): continue
    with open(f'{DATA_DIR}/labels/train/{lf}') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                widths.append(float(parts[3]))
                heights.append(float(parts[4]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=30, color='steelblue', edgecolor='white')
axes[0].axvline(x=0.8, color='red', linestyle='--', label='>0.8 suspicious')
axes[0].set_xlabel('Normalized box width'); axes[0].set_title('Box width distribution')
axes[0].legend()
axes[1].hist(heights, bins=30, color='darkorange', edgecolor='white')
axes[1].axvline(x=0.8, color='red', linestyle='--', label='>0.8 suspicious')
axes[1].set_xlabel('Normalized box height'); axes[1].set_title('Box height distribution')
axes[1].legend()
plt.suptitle('VIZ 1.C — Bounding box size distribution')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/viz1c_box_sizes.png', dpi=150)
plt.show()

print(f'Total annotations: {len(widths)}')
print(f'Boxes width  > 0.8: {sum(w>0.8 for w in widths)}  ← annotation errors if high')
print(f'Boxes height > 0.8: {sum(h>0.8 for h in heights)}  ← annotation errors if high')
print(f'Boxes width  < 0.02: {sum(w<0.02 for w in widths)}  ← too small to be useful')

## Phase 1 Completion Checklist

Before moving to Phase 2, verify:

- [ ] ≥ 300 total images (train + val + test)
- [ ] All images annotated in YOLO format
- [ ] Train/val/test/calibration split done
- [ ] Calibration set: exactly 100 images carved from train
- [ ] **VIZ 1.A saved** — all class bars ≥ 25
- [ ] **VIZ 1.B saved** — boxes visually correct
- [ ] **VIZ 1.C saved** — no mass of boxes > 0.8
- [ ] 100% annotation coverage (every image has a label file)

If all checked, open `02_baseline_train.ipynb`.